In [1]:
"""
✅ QWEN2.5-7B FEW-SHOT RHETORICAL ROLE CLASSIFIER
✅ Dynamic few-shot examples from training dataset
✅ 13-class rhetorical roles with comprehensive evaluation
✅ Full metrics: Accuracy, Precision, Recall, F1 (Macro & Weighted), Confusion Matrix
✅ Production-ready for arXiv publication
"""

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from transformers import pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, 
    f1_score, precision_score, recall_score
)
from collections import defaultdict
import random
import warnings
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
DATA_PATHS = {
    "train": "build_jsonl/build_train.jsonl",
    "dev": "build_jsonl/build_dev.jsonl", 
    "test": "build_jsonl/build_test.jsonl"
}
OUT_DIR = "qwen_fewshot_results"
os.makedirs(OUT_DIR, exist_ok=True)

LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT", 
    "ANALYSIS", "STA", "PRE_RELIED", "PRE_NOT_RELIED", "RATIO", "RPC", "NONE"
]
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(LABELS)

# Number of few-shot examples per class
EXAMPLES_PER_CLASS = 3

# ---------------- RHETORICAL ROLE DEFINITIONS ----------------
ROLE_DEFINITIONS = {
    "PREAMBLE": "Introductory metadata: court name, case number, parties, judges, hearing date",
    "FAC": "Factual background: chronological events, circumstances, and case history", 
    "RLC": "Rulings by lower courts: trial court or high court decisions and verdicts",
    "ISSUE": "Legal questions or issues framed for adjudication by the court",
    "ARG_PETITIONER": "Arguments and submissions presented by petitioner/appellant",
    "ARG_RESPONDENT": "Counter-arguments and submissions by respondent/defendant",
    "ANALYSIS": "Court's legal reasoning: interpretation and application of facts and law",
    "STA": "Statutory provisions: specific sections, acts, rules, regulations cited",
    "PRE_RELIED": "Precedents followed: binding legal authorities and case laws relied upon",
    "PRE_NOT_RELIED": "Precedents distinguished: case laws not followed or differentiated",
    "RATIO": "Core legal principle: ratio decidendi - the binding legal reasoning",
    "RPC": "Final ruling and relief: appeal allowed/dismissed, orders, directions given",
    "NONE": "Non-rhetorical content: headings, page numbers, table of contents, formatting"
}

# ---------------- DATA LOADING ----------------
def load_jsonl(path):
    """Load JSONL dataset"""
    if not os.path.exists(path):
        print(f"⚠️  Dataset not found: {path}")
        return []
    
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if s:
                try:
                    data.append(json.loads(s))
                except json.JSONDecodeError:
                    continue
    return data

def extract_data(docs, max_sents=64):
    """Extract sentences and labels from documents"""
    all_sents, all_labels, doc_ids = [], [], []
    
    for doc in docs:
        doc_id = doc.get("id", "unknown")
        sents, labs = [], []
        
        # Handle different JSON formats
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs = doc["labels"]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs = doc["annotation"]
        
        # Limit sentence count
        if len(sents) > max_sents:
            sents = sents[:max_sents]
            labs = labs[:max_sents]
        
        if sents and labs and len(sents) == len(labs):
            all_sents.append(sents)
            all_labels.append(labs)
            doc_ids.append(doc_id)
    
    return all_sents, all_labels, doc_ids

def extract_few_shot_examples(train_docs, examples_per_class=3):
    """Extract balanced few-shot examples from training dataset"""
    print(f"🔍 Extracting {examples_per_class} examples per class from training data...")
    
    # Collect examples by class
    class_examples = defaultdict(list)
    
    for doc in train_docs:
        sents, labs = [], []
        if "sentences" in doc and "labels" in doc:
            sents = doc["sentences"]
            labs = doc["labels"]
        elif "sentences" in doc and "annotation" in doc:
            sents = doc["sentences"]
            labs = doc["annotation"]
        
        for sent, label in zip(sents, labs):
            if label in LABELS and len(sent.strip()) > 10:  # Filter short sentences
                class_examples[label].append(sent)
    
    # Sample balanced examples
    few_shot_examples = []
    for label in LABELS:
        if label in class_examples and len(class_examples[label]) > 0:
            # Sample diverse examples
            available = class_examples[label]
            random.seed(42)  # For reproducibility
            sampled = random.sample(available, min(examples_per_class, len(available)))
            
            for sent in sampled:
                few_shot_examples.append({"text": sent, "label": label})
    
    print(f"✅ Extracted {len(few_shot_examples)} total examples across {len(set([ex['label'] for ex in few_shot_examples]))} classes")
    
    # Print distribution
    label_counts = defaultdict(int)
    for ex in few_shot_examples:
        label_counts[ex['label']] += 1
    print("\n📊 Few-shot distribution:")
    for label in LABELS:
        print(f"  {label}: {label_counts[label]} examples")
    
    return few_shot_examples

# ---------------- QWEN FEW-SHOT PREDICTOR ----------------
class QwenFewShotClassifier:
    def __init__(self, few_shot_examples):
        self.pipe = None
        self.few_shot_examples = few_shot_examples
        
    def load_model(self):
        """Load Qwen2.5-7B-Instruct model"""
        print("\n🚀 Loading Qwen2.5-7B-Instruct...")
        self.pipe = pipeline(
            "text-generation",
            model="Qwen/Qwen2.5-7B-Instruct",
            torch_dtype=torch.float16,
            device_map="auto",
            max_new_tokens=20,
            do_sample=False,
            temperature=0.01,
            top_p=0.95,
            pad_token_id=151643,  # Qwen EOS token
            trust_remote_code=True
        )
        print("✅ Qwen2.5-7B loaded successfully!")
    
    def create_prompt(self, sentence):
        """Create enhanced few-shot prompt optimized for Qwen"""
        prompt = "<|im_start|>system\nYou are an expert legal document classifier for Indian Supreme Court judgments. Classify sentences into exactly ONE of 13 rhetorical roles. Respond with ONLY the label name.\n<|im_end|>\n"
        
        prompt += "<|im_start|>user\n"
        prompt += "CLASSIFICATION TASK (13 classes):\n\n"
        prompt += "LABELS:\n" + "\n".join([f"- {label}" for label in LABELS]) + "\n\n"
        
        prompt += "DEFINITIONS:\n"
        for label, definition in ROLE_DEFINITIONS.items():
            prompt += f"- {label}: {definition}\n"
        
        prompt += "\nFEW-SHOT EXAMPLES:\n"
        for ex in self.few_shot_examples:
            prompt += f"Text: \"{ex['text'][:100]}...\"\nLabel: {ex['label']}\n\n"
        
        prompt += "TO CLASSIFY:\n"
        prompt += f"Text: \"{sentence}\"\n"
        prompt += "Label: <|im_end|>\n"
        prompt += "<|im_start|>assistant\n"
        
        return prompt
    
    def parse_response(self, text):
        """Extract label from Qwen response with robust parsing"""
        text = text.strip().upper()
        
        # Direct label match (prioritize exact matches)
        for label in LABELS:
            if label in text and len(text) < 20:  # Short responses likely exact
                return label
        
        # Substring match
        for label in LABELS:
            if label in text:
                return label
        
        # Qwen-specific patterns (stronger legal reasoning)
        patterns = {
            "PREAMBLE": ["CASE NO", "CORAM", "BENCH", "HEARD ON"],
            "FAC": ["FACTS", "BACKGROUND", "EVENTS", "CIRCUMSTANCES"],
            "RLC": ["LOWER COURT", "TRIAL COURT", "HIGH COURT", "HELD THAT"],
            "ISSUE": ["ISSUE", "QUESTION", "FRAMED", "FOR CONSIDERATION"],
            "ARG_PETITIONER": ["PETITIONER SUBMITS", "APPELLANT CONTENDS", "SUBMISSION IS"],
            "ARG_RESPONDENT": ["RESPONDENT SUBMITS", "DEFENDANT ARGUES"],
            "ANALYSIS": ["WE ARE OF THE VIEW", "CONSIDERING", "IN LIGHT OF"],
            "STA": ["SECTION", "ACT", "RULE", "PROVISO"],
            "PRE_RELIED": ["FOLLOWED", "APPROVED", "REITERATED", "(AIR)"],
            "PRE_NOT_RELIED": ["DISTINGUISHED", "NOT APPLICABLE"],
            "RATIO": ["RATIO", "PRINCIPLE", "LAID DOWN"],
            "RPC": ["HELD", "APPEAL ALLOWED", "DISMISSED", "DIRECTED"],
            "NONE": ["CONTENTS", "INDEX", "PAGE"]
        }
        
        text_lower = text.lower()
        for label, keywords in patterns.items():
            if any(kw.lower() in text_lower for kw in keywords):
                return label
        
        return "NONE"
    
    def predict(self, sentences, batch_size=1):
        """Predict rhetorical roles for sentences"""
        print(f"\n🔮 Predicting {len(sentences)} sentences with Qwen2.5-7B Few-Shot...")
        predictions = []
        
        for i, sent in enumerate(sentences):
            if i % 100 == 0:
                print(f"  Progress: {i}/{len(sentences)} sentences...")
            
            try:
                prompt = self.create_prompt(sent)
                response = self.pipe(prompt, return_full_text=False)[0]['generated_text']
                pred = self.parse_response(response)
                predictions.append(pred)
            except Exception as e:
                print(f"  ⚠️  Error at sentence {i}: {e}")
                predictions.append("NONE")
        
        print(f"✅ Prediction complete: {len(predictions)} labels generated")
        return predictions

# ---------------- EVALUATION METRICS ----------------
def compute_comprehensive_metrics(y_true, y_pred):
    """Compute all evaluation metrics"""
    # Convert to numeric
    y_true_num = np.array([label2id[l] for l in y_true])
    y_pred_num = np.array([label2id.get(p, label2id["NONE"]) for p in y_pred])
    
    # Overall metrics
    accuracy = accuracy_score(y_true_num, y_pred_num)
    
    # Precision, Recall, F1 (Macro)
    precision_macro = precision_score(y_true_num, y_pred_num, average='macro', zero_division=0)
    recall_macro = recall_score(y_true_num, y_pred_num, average='macro', zero_division=0)
    f1_macro = f1_score(y_true_num, y_pred_num, average='macro', zero_division=0)
    
    # Precision, Recall, F1 (Weighted)
    precision_weighted = precision_score(y_true_num, y_pred_num, average='weighted', zero_division=0)
    recall_weighted = recall_score(y_true_num, y_pred_num, average='weighted', zero_division=0)
    f1_weighted = f1_score(y_true_num, y_pred_num, average='weighted', zero_division=0)
    
    # Per-class metrics
    report = classification_report(
        y_true_num, y_pred_num, 
        labels=list(range(NUM_LABELS)),
        target_names=LABELS,
        digits=4, 
        zero_division=0,
        output_dict=True
    )
    
    report_str = classification_report(
        y_true_num, y_pred_num, 
        labels=list(range(NUM_LABELS)),
        target_names=LABELS,
        digits=4, 
        zero_division=0
    )
    
    metrics = {
        'accuracy': accuracy,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        'classification_report': report,
        'classification_report_str': report_str,
        'y_true_num': y_true_num,
        'y_pred_num': y_pred_num
    }
    
    return metrics

def save_results(metrics, predictions, true_labels, test_sentences):
    """Save all results and visualizations"""
    print("\n💾 Saving results...")
    
    # 1. Overall metrics summary
    summary_df = pd.DataFrame({
        'Metric': [
            'Accuracy', 
            'Precision (Macro)', 'Recall (Macro)', 'F1-Score (Macro)',
            'Precision (Weighted)', 'Recall (Weighted)', 'F1-Score (Weighted)'
        ],
        'Value': [
            metrics['accuracy'],
            metrics['precision_macro'], metrics['recall_macro'], metrics['f1_macro'],
            metrics['precision_weighted'], metrics['recall_weighted'], metrics['f1_weighted']
        ]
    })
    summary_df.to_csv(f'{OUT_DIR}/overall_metrics.csv', index=False)
    print(f"✅ Saved: {OUT_DIR}/overall_metrics.csv")
    
    # 2. Per-class metrics
    report_dict = metrics['classification_report']
    per_class_data = []
    for label in LABELS:
        if label in report_dict:
            per_class_data.append({
                'Class': label,
                'Precision': report_dict[label]['precision'],
                'Recall': report_dict[label]['recall'],
                'F1-Score': report_dict[label]['f1-score'],
                'Support': report_dict[label]['support']
            })
    
    per_class_df = pd.DataFrame(per_class_data)
    per_class_df.to_csv(f'{OUT_DIR}/per_class_metrics.csv', index=False)
    print(f"✅ Saved: {OUT_DIR}/per_class_metrics.csv")
    
    # 3. Detailed classification report
    with open(f'{OUT_DIR}/classification_report.txt', 'w') as f:
        f.write("QWEN2.5-7B FEW-SHOT CLASSIFICATION REPORT\n")
        f.write("="*80 + "\n\n")
        f.write(metrics['classification_report_str'])
    print(f"✅ Saved: {OUT_DIR}/classification_report.txt")
    
    # 4. Predictions CSV
    predictions_df = pd.DataFrame({
        'Sentence': test_sentences[:len(predictions)],
        'True_Label': true_labels[:len(predictions)],
        'Predicted_Label': predictions
    })
    predictions_df.to_csv(f'{OUT_DIR}/predictions.csv', index=False)
    print(f"✅ Saved: {OUT_DIR}/predictions.csv")
    
    # 5. Confusion Matrix (Heatmap)
    cm = confusion_matrix(metrics['y_true_num'], metrics['y_pred_num'], labels=range(NUM_LABELS))
    
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=LABELS, yticklabels=LABELS,
                cbar_kws={'label': 'Count'}, linewidths=0.5)
    plt.title('Qwen2.5-7B Few-Shot Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
    plt.xlabel('Predicted Label', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {OUT_DIR}/confusion_matrix.png")
    
    # 6. Per-class F1 scores (Bar plot)
    plt.figure(figsize=(14, 8))
    f1_scores = [report_dict[label]['f1-score'] for label in LABELS if label in report_dict]
    colors = plt.cm.plasma(np.linspace(0, 1, len(LABELS)))
    
    bars = plt.bar(LABELS, f1_scores, color=colors, edgecolor='black', linewidth=1.2)
    plt.axhline(y=metrics['f1_macro'], color='red', linestyle='--', 
                linewidth=2, label=f'Macro F1: {metrics["f1_macro"]:.4f}')
    plt.axhline(y=metrics['f1_weighted'], color='blue', linestyle='--', 
                linewidth=2, label=f'Weighted F1: {metrics["f1_weighted"]:.4f}')
    
    plt.xlabel('Rhetorical Role', fontsize=14, fontweight='bold')
    plt.ylabel('F1-Score', fontsize=14, fontweight='bold')
    plt.title('Per-Class F1-Scores - Qwen2.5-7B Few-Shot', fontsize=16, fontweight='bold', pad=20)
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1.0)
    plt.legend(fontsize=12, loc='upper right')
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/f1_scores_per_class.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {OUT_DIR}/f1_scores_per_class.png")
    
    # 7. Metrics comparison (Precision, Recall, F1)
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    metrics_to_plot = ['precision', 'recall', 'f1-score']
    titles = ['Precision', 'Recall', 'F1-Score']
    
    for idx, (metric, title) in enumerate(zip(metrics_to_plot, titles)):
        values = [report_dict[label][metric] for label in LABELS if label in report_dict]
        axes[idx].bar(LABELS, values, color=colors, edgecolor='black', linewidth=1.2)
        axes[idx].set_xlabel('Rhetorical Role', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel(title, fontsize=12, fontweight='bold')
        axes[idx].set_title(f'{title} by Class', fontsize=14, fontweight='bold')
        axes[idx].tick_params(axis='x', rotation=45)
        axes[idx].set_ylim(0, 1.0)
        axes[idx].grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(f'{OUT_DIR}/precision_recall_f1_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    print(f"✅ Saved: {OUT_DIR}/precision_recall_f1_comparison.png")

# ---------------- MAIN EXECUTION ----------------
def main():
    print("="*80)
    print("🎯 QWEN2.5-7B FEW-SHOT RHETORICAL ROLE CLASSIFIER")
    print("="*80)
    print("📋 Task: 13-class Rhetorical Role Classification")
    print("🧠 Model: Qwen/Qwen2.5-7B-Instruct")
    print("🎓 Learning: Few-Shot with Training Examples")
    print("="*80)
    
    # 1. Load datasets
    print("\n📂 Loading datasets...")
    train_docs = load_jsonl(DATA_PATHS["train"])
    test_docs = load_jsonl(DATA_PATHS["test"])
    print(f"✅ Loaded {len(train_docs)} training documents")
    print(f"✅ Loaded {len(test_docs)} test documents")
    
    # 2. Extract few-shot examples from training data
    few_shot_examples = extract_few_shot_examples(train_docs, EXAMPLES_PER_CLASS)
    
    # 3. Extract test data
    test_sents, test_labels, test_doc_ids = extract_data(test_docs)
    
    # Flatten for evaluation (limit to first 500 sentences for efficiency)
    flat_test_sents = []
    flat_test_labels = []
    for doc_sents, doc_labels in zip(test_sents[:50], test_labels[:50]):
        for sent, label in zip(doc_sents[:15], doc_labels[:15]):
            flat_test_sents.append(sent)
            flat_test_labels.append(label)
    
    print(f"\n📊 Test dataset: {len(flat_test_sents)} sentences")
    
    # 4. Initialize and load classifier
    classifier = QwenFewShotClassifier(few_shot_examples)
    classifier.load_model()
    
    # 5. Make predictions
    predictions = classifier.predict(flat_test_sents)
    
    # 6. Compute metrics
    print("\n📈 Computing evaluation metrics...")
    metrics = compute_comprehensive_metrics(flat_test_labels, predictions)
    
    # 7. Display results
    print("\n" + "="*80)
    print("📊 EVALUATION RESULTS")
    print("="*80)
    print(f"\n{'Metric':<30} {'Value':>15}")
    print("-"*50)
    print(f"{'Accuracy':<30} {metrics['accuracy']:>15.4f}")
    print(f"{'Precision (Macro)':<30} {metrics['precision_macro']:>15.4f}")
    print(f"{'Recall (Macro)':<30} {metrics['recall_macro']:>15.4f}")
    print(f"{'F1-Score (Macro)':<30} {metrics['f1_macro']:>15.4f}")
    print(f"{'Precision (Weighted)':<30} {metrics['precision_weighted']:>15.4f}")
    print(f"{'Recall (Weighted)':<30} {metrics['recall_weighted']:>15.4f}")
    print(f"{'F1-Score (Weighted)':<30} {metrics['f1_weighted']:>15.4f}")
    print("="*80)
    
    print("\n📋 DETAILED CLASSIFICATION REPORT:")
    print(metrics['classification_report_str'])
    
    # 8. Save all results
    save_results(metrics, predictions, flat_test_labels, flat_test_sents)
    
    # 9. Final summary
    print("\n" + "="*80)
    print("🎉 EVALUATION COMPLETE!")
    print("="*80)
    print(f"📁 All results saved to: {OUT_DIR}/")
    print("\n📄 Generated files:")
    print("  ✅ overall_metrics.csv - Summary metrics")
    print("  ✅ per_class_metrics.csv - Class-wise performance")
    print("  ✅ classification_report.txt - Detailed report")
    print("  ✅ predictions.csv - All predictions with sentences")
    print("  ✅ confusion_matrix.png - Confusion matrix heatmap")
    print("  ✅ f1_scores_per_class.png - F1 scores bar chart")
    print("  ✅ precision_recall_f1_comparison.png - Metrics comparison")
    print("="*80)
    print("\n🚀 Ready for arXiv publication!")
    print("="*80)

if __name__ == "__main__":
    main()


🎯 QWEN2.5-7B FEW-SHOT RHETORICAL ROLE CLASSIFIER
📋 Task: 13-class Rhetorical Role Classification
🧠 Model: Qwen/Qwen2.5-7B-Instruct
🎓 Learning: Few-Shot with Training Examples

📂 Loading datasets...
✅ Loaded 245 training documents
✅ Loaded 50 test documents
🔍 Extracting 3 examples per class from training data...
✅ Extracted 39 total examples across 13 classes

📊 Few-shot distribution:
  PREAMBLE: 3 examples
  FAC: 3 examples
  RLC: 3 examples
  ISSUE: 3 examples
  ARG_PETITIONER: 3 examples
  ARG_RESPONDENT: 3 examples
  ANALYSIS: 3 examples
  STA: 3 examples
  PRE_RELIED: 3 examples
  PRE_NOT_RELIED: 3 examples
  RATIO: 3 examples
  RPC: 3 examples
  NONE: 3 examples

📊 Test dataset: 750 sentences

🚀 Loading Qwen2.5-7B-Instruct...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


✅ Qwen2.5-7B loaded successfully!

🔮 Predicting 750 sentences with Qwen2.5-7B Few-Shot...
  Progress: 0/750 sentences...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


  Progress: 100/750 sentences...
  Progress: 200/750 sentences...
  Progress: 300/750 sentences...
  Progress: 400/750 sentences...
  Progress: 500/750 sentences...
  Progress: 600/750 sentences...
  Progress: 700/750 sentences...
✅ Prediction complete: 750 labels generated

📈 Computing evaluation metrics...

📊 EVALUATION RESULTS

Metric                                   Value
--------------------------------------------------
Accuracy                                0.4787
Precision (Macro)                       0.1754
Recall (Macro)                          0.2666
F1-Score (Macro)                        0.1690
Precision (Weighted)                    0.8275
Recall (Weighted)                       0.4787
F1-Score (Weighted)                     0.5770

📋 DETAILED CLASSIFICATION REPORT:
                precision    recall  f1-score   support

      PREAMBLE     0.9481    0.4337    0.5951       505
           FAC     0.7383    0.6011    0.6627       183
           RLC     0.1552    0.5625 